In [5]:
import pandas as pd
from app.retrieve import search_similar_chunks
from app.generate import generate_answer

In [6]:
import inspect
print(inspect.signature(search_similar_chunks))

(query: str, top_k: int = 5, initial_k: int = 50, max_chunks_per_doc: int = 1, vector_weight: float = 0.5, keyword_weight: float = 0.5)


In [7]:
test_questions = [
    {
        "question": "Which markets show strong rental demand?",
        "expected_docs": ["sample_report.txt", "phoenix_report.txt", "dallas_report.txt"]
    },
    {
        "question": "Which market mentions insurance costs?",
        "expected_docs": ["dallas_report.txt"]
    },
    {
        "question": "Which market mentions in-migration?",
        "expected_docs": ["phoenix_report.txt"]
    },
    {
        "question": "What affordability risks are mentioned?",
        "expected_docs": ["sample_report.txt", "phoenix_report.txt", "dallas_report.txt"]
    }
]

In [3]:
results = search_similar_chunks(
    "Which market mentions insurance costs?",
    top_k=10,
    initial_k=50,
    max_chunks_per_doc=2,
    vector_weight=0.5,
    keyword_weight=0.5
)

for r in results:
    print(
        r["document_name"],
        "| hybrid =", round(r["hybrid_score"], 4),
        "| vector_distance =", round(r["vector_distance"], 4),
        "| keyword_score =", round(r["keyword_score"], 4),
        "| page =", r["page_number"],
        "| chunk =", r["chunk_id"]
    )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


dallas_report.txt | hybrid = 0.7882 | vector_distance = 0.7351 | keyword_score = 2.0 | page = None | chunk = 0
zillow_report.pdf | hybrid = 0.5392 | vector_distance = 0.729 | keyword_score = 1.0 | page = 55 | chunk = 2
zillow_report.pdf | hybrid = 0.3006 | vector_distance = 0.6634 | keyword_score = 0.0 | page = 42 | chunk = 1
phoenix_report.txt | hybrid = 0.2859 | vector_distance = 0.7486 | keyword_score = 0.0 | page = None | chunk = 0


In [11]:
rows = []

for item in test_questions:
    results = search_similar_chunks(
        item["question"],
        top_k=3,
        initial_k=50,
        max_chunks_per_doc=1,
        vector_weight=0.5,
        keyword_weight=0.5
    )
    retrieved_docs = [r["document_name"] for r in results]
    hit = any(doc in item["expected_docs"] for doc in retrieved_docs[:3])

    rows.append({
        "question": item["question"],
        "expected_docs": ", ".join(item["expected_docs"]),
        "retrieved_top3": ", ".join(retrieved_docs[:3]),
        "hit_at_3": hit
    })

df = pd.DataFrame(rows)
df

,question,expected_docs,retrieved_top3,hit_at_3
0,Which markets show strong rental demand?,"sample_report.txt, phoenix_report.txt, dallas_...","phoenix_report.txt, sample_report.txt, zillow_...",True
1,Which market mentions insurance costs?,dallas_report.txt,"dallas_report.txt, zillow_report.pdf, phoenix_...",True
2,Which market mentions in-migration?,phoenix_report.txt,zillow_report.pdf,False
3,What affordability risks are mentioned?,"sample_report.txt, phoenix_report.txt, dallas_...","zillow_report.pdf, sample_report.txt, phoenix_...",True


In [12]:
hit_rate = df["hit_at_3"].mean()
print(f"Hit@3: {hit_rate:.2%}")

Hit@3: 75.00%


In [14]:
question = "Which markets show strong rental demand?"
results = search_similar_chunks(
    question,
    top_k=3,
    initial_k=50,
    max_chunks_per_doc=1,
    vector_weight=0.5,
    keyword_weight=0.5
)
answer = generate_answer(question, results)

print("Question:", question)
print("\nAnswer:\n", answer)
print("\nSources:")
for r in results:
    print(
        f"- {r['document_name']} | "
        f"page={r['page_number']} | "
        f"chunk={r['chunk_id']} | "
        f"hybrid_score={float(r['hybrid_score']):.4f}"
    )

Question: Which markets show strong rental demand?

Answer:
 The markets showing strong rental demand are Phoenix and Austin. The Phoenix housing market continued to have strong rental demand in 2025, supported by in-migration, employment growth, and limited supply. The Austin housing market also showed steady rental demand in 2025, supported by population growth, job creation, and limited housing inventory.

Sources:
- phoenix_report.txt | page=None | chunk=0 | hybrid_score=0.8492
- sample_report.txt | page=None | chunk=0 | hybrid_score=0.8488
- zillow_report.pdf | page=69 | chunk=2 | hybrid_score=0.5128


In [15]:
df

,question,expected_docs,retrieved_top3,hit_at_3
0,Which markets show strong rental demand?,"sample_report.txt, phoenix_report.txt, dallas_...","phoenix_report.txt, sample_report.txt, zillow_...",True
1,Which market mentions insurance costs?,dallas_report.txt,"dallas_report.txt, zillow_report.pdf, phoenix_...",True
2,Which market mentions in-migration?,phoenix_report.txt,zillow_report.pdf,False
3,What affordability risks are mentioned?,"sample_report.txt, phoenix_report.txt, dallas_...","zillow_report.pdf, sample_report.txt, phoenix_...",True


In early evaluation, retrieval performance dropped when a large multi-page PDF was added to a small document set. After removing the oversized document, Hit@3 improved from 25% to 100%, showing that corpus balance and chunk distribution strongly affect retrieval quality.

Retrieval evaluation showed that adding a large multi-page PDF reduced Hit@3 from 100% to 25% on a small benchmark. After adding document-diverse retrieval, Hit@3 improved to 50%, showing that source balancing can reduce domination by oversized documents.
